# 2. Model Training
**Models**: LSTM, GRU, LSTMCNN, GRUCNN

Training with early stopping, window size = 10, batch size = 64


In [ ]:
import sys, os, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import joblib

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
np.random.seed(42)
torch.manual_seed(42)

# Load processed data from Notebook 1
train_df = pd.read_csv('results/train_data.csv')
val_df = pd.read_csv('results/val_data.csv')
test_df = pd.read_csv('results/test_data.csv')
scaler = joblib.load('results/scaler.save')

with open('results/config.json', 'r') as f:
    config = json.load(f)
feature_cols = config['feature_cols']
target_col = config['target_col']
WINDOW_SIZE = config['window_size']

print(f"\nTrain: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")
print(f"Features: {len(feature_cols)}, Window: {WINDOW_SIZE}")


## 2.1 Model Architectures

| Model | Architecture |
|---|---|
| **LSTM** | LSTM(hidden=10, layers=3) -> Linear(10,1) |
| **GRU** | GRU(hidden=10, layers=3) -> Linear(10,1) |
| **LSTMCNN** | LSTM(hidden=16, layers=1) -> Conv1d(16,8,k=1) -> MaxPool(4) -> FC(16) -> FC(1) |
| **GRUCNN** | GRU(hidden=16, layers=1) -> Conv1d(16,8,k=1) -> MaxPool(4) -> FC(16) -> FC(1) |


In [ ]:
from models import get_model

input_size = len(feature_cols)
for name in ['LSTM', 'GRU', 'LSTMCNN', 'GRUCNN']:
    model = get_model(name, input_size, WINDOW_SIZE)
    total_params = sum(p.numel() for p in model.parameters())
    print(f"\n{'='*50}")
    print(f"{name} ({total_params:,} parameters)")
    print(f"{'='*50}")
    print(model)


## 2.2 Training Configuration

| Parameter | Value |
|---|---|
| Batch size | 64 |
| Learning rate | 0.001 |
| Optimizer | Adam |
| Loss function | MSE |
| Max epochs | 100 |
| Early stopping patience | 10 |
| Window size | 10 |


## 2.3 Train All Models

In [ ]:
from train import train_all_models

results = train_all_models(
    train_df=train_df,
    val_df=val_df,
    test_df=test_df,
    feature_cols=feature_cols,
    target_col=target_col,
    window_size=WINDOW_SIZE,
    batch_size=64,
    lr=0.001,
    epochs=100,
    patience=10,
    save_dir='results',
    device=DEVICE,
    scaler=scaler
)

print("\n[OK] All 4 models trained successfully!")


## 2.4 Training Loss Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Training & Validation Loss Curves', fontsize=16, fontweight='bold')
colors = {'LSTM': '#FF6B6B', 'GRU': '#4ECDC4', 'LSTMCNN': '#45B7D1', 'GRUCNN': '#96CEB4'}

for ax, (name, data) in zip(axes.flatten(), results.items()):
    ax.plot(data['train_losses'], label='Train', color=colors[name], linewidth=2, alpha=0.8)
    ax.plot(data['val_losses'], label='Validation', color='orange', linewidth=2, alpha=0.8)
    ax.set_title(name, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/plots/training_loss.png', dpi=150, bbox_inches='tight')
plt.show()


## 2.5 Save Predictions

In [ ]:
import json as json2

os.makedirs('results/predictions', exist_ok=True)

for name, data in results.items():
    # Save predictions
    pred_df = pd.DataFrame({
        'prediction': data['test_pred'],
        'actual': data['test_actual']
    })
    pred_df.to_csv(f'results/predictions/{name}_predictions.csv', index=False)
    
    # Save loss history
    loss_df = pd.DataFrame({
        'train_loss': data['train_losses'],
        'val_loss': data['val_losses']
    })
    loss_df.to_csv(f'results/predictions/{name}_losses.csv', index=False)
    
    print(f"  {name}: {len(data['test_pred'])} predictions saved, {len(data['train_losses'])} epochs")

print("\n[OK] All predictions and loss histories saved!")
